# 🌿 Plant Disease Detection

A production-grade, multilingual plant disease detection and treatment advisory app built for Indian farmers.

---

| Feature | Detail |
|---------|--------|
| **Vision Models** | YOLOv8n & YOLOv12n (fine-tuned on 28-class dataset) |
| **Ensemble Logic** | Confidence-threshold agreement / disagreement routing |
| **Languages** | English · हिन्दी · اردو · తెలుగు |
| **Translation** | IndicTrans2 (`ai4bharat/indictrans2-en-indic-dist-200M`) |
| **Treatment RAG** | Google Gemini 2.0 Flash + ChromaDB vector store |
| **TTS** | `edge-tts` neural voices per language |
| **Fallback** | Symptom-based AI diagnosis when confidence < 50% |

---

## 📁 File Structure

```
plant_disease_app/
├── app.py                  ← Main Gradio application
├── translations.py         ← UI strings + disease taxonomy (all 4 languages)
├── disease_id_mapping.py   ← YOLO class → disease_id mapping
├── treatment_data.py       ← Inline treatment database (all 28 diseases)
├── data_split.yaml         ← Dataset config reference
├── requirements.txt
├── models/
│   ├── finetuned_yolo8n.pt   ← Upload your fine-tuned YOLOv8n weights here
│   └── finetuned_yolo12n.pt  ← Upload your fine-tuned YOLOv12n weights here
└── README.md
```

---

## ⚙️ Setup on Hugging Face Spaces

### Step 1 — Create the Space

1. Go to [huggingface.co/spaces](https://huggingface.co/spaces) → **Create new Space**
2. Set SDK → **Gradio**, Hardware → **CPU (free)** or **T4 GPU** (recommended)
3. Clone the Space repo:
   ```bash
   git clone https://huggingface.co/spaces/YOUR_USERNAME/plant-disease-detector
   cd plant-disease-detector
   ```

### Step 2 — Upload all project files

Copy all files from this folder into the cloned Space repo:
```bash
cp -r /path/to/plant_disease_app/* .
mkdir -p models
# Copy your trained weights:
cp /path/to/finetuned_yolo8n.pt  models/
cp /path/to/finetuned_yolo12n.pt models/
```

### Step 3 — Install IndicTransToolkit (via packages.txt)

Create a `packages.txt` file in the Space root:
```
git
```

Then add this to a `setup.sh` or as a `pre_install` step. The toolkit is installed automatically via the git dependency in requirements.txt using:
```
# Add this line to requirements.txt:
git+https://github.com/VarunGumma/IndicTransToolkit.git
```

> **Note:** Add the git line directly to requirements.txt — HF Spaces supports `git+https://` installs natively.

### Step 4 — Set Secret Environment Variables

In your Space → **Settings → Repository secrets**, add:

| Secret Name | Value |
|-------------|-------|
| `HF_TOKEN` | Your Hugging Face read token (for IndicTrans2 model download) |
| `GOOGLE_API_KEY` | Your Google AI Studio API key (for Gemini RAG) |
| `YOLO_V8_PATH` | `models/finetuned_yolo8n.pt` (default, change if needed) |
| `YOLO_V12_PATH` | `models/finetuned_yolo12n.pt` |

### Step 5 — Push and deploy

```bash
git add .
git commit -m "Initial deployment"
git push
```

The Space will build automatically. First startup takes ~3–5 minutes (downloading IndicTrans2).

---

## 🔬 Supported Crops & Diseases (28 Classes)

| Crop | Disease Classes |
|------|----------------|
| 🍎 Apple | Black Rot, Cedar Rust, Scab, Healthy |
| 🌽 Corn/Maize | Common Rust, Gray Leaf Spot, Leaf Blight, Northern Leaf Blight, Healthy |
| 🍇 Grape | Black Rot, Esca, Leaf Blight, Healthy |
| 🥔 Potato | Early Blight, Late Blight, Healthy |
| 🍅 Tomato | Bacterial Spot, Bacterial Wilt, Brown Spots, Early Blight, Late Blight, Leaf Mold, Mosaic Virus, Septoria Leaf Spot, Spider Mites, Target Spot, Yellow Leaf Curl Virus, Healthy |

---

## 🏗️ Architecture

```
User uploads image
        │
        ▼
  ┌─────────────────────┐
  │  YOLOv8n  (640×640) │───────┐
  └─────────────────────┘       │
                                ▼

  ┌─────────────────────┐   Ensemble
  │  YOLOv12n (640×640) │──── Logic ──► Confidence ≥ 50%?
  └─────────────────────┘               │              │
                                        YES            NO
                                        │              │
                               ┌────────┘    ┌─────────────────┐
                               │             │  Symptom Fallback│
                               │             │  (Gemini RAG)    │
                               ▼             └─────────────────┘
                    ┌──────────────────┐
                    │  Disease Info    │  ← translations.py
                    │  (28 classes)    │
                    └──────────────────┘
                               │
                    ┌──────────┴──────────┐
                    │                     │
           Disease Details          Treatment Plan
          (IndicTrans2 TL)         (TREATMENT_DB + RAG)
                    │                     │
                    └──────────┬──────────┘
                               │
                          edge-tts Audio
                         (4 languages)
```

---

## 📝 Licenses & Credits

- **YOLOv8/v12:** Ultralytics AGPL-3.0
- **IndicTrans2:** CC-BY 4.0 (AI4Bharat)
- **Gemini API:** Google Terms of Service
- **Treatment Data:** Compiled from ICAR, TNAU, KVK, UC IPM public resources

---

*Built with ❤️ for Indian agriculture. For crop advisory, always consult your local KVK officer.*

In [1]:
# ── Cell 1: Install all dependencies ─────────────────────────────────────────
import subprocess, sys

packages = [
    'ultralytics>=8.3.0',
    'Pillow>=10.0.0',
    'opencv-python-headless>=4.9.0',
    'transformers>=4.41.0,<5.0.0',
    'sentencepiece>=0.2.0',
    'langchain>=0.3.0',
    'langchain-core>=0.3.0',
    'langchain-community>=0.3.0',
    'langchain-google-genai>=2.0.0',
    'langchain-huggingface>=0.1.0',
    'langchain-chroma>=0.1.0',
    'chromadb>=0.5.0',
    'sentence-transformers>=3.0.0',
    'edge-tts>=6.1.10',
    'huggingface_hub>=0.24.0',
    'gradio>=5.20.0',
    'pyyaml>=6.0',
    'requests>=2.31.0',
    'tqdm>=4.66.0',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('✅ All packages installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 732.2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.49.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-adk 1.21.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.21.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.40.0 which is in

✅ All packages installed.


In [2]:
# ── Cell 2: Set environment variables ────────────────────────────────────────
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# ── Retrieve secrets ──────────────────────────────────────────────────────────
# Add these in Kaggle: Add-ons → Secrets → Add Secret
try:
    os.environ['HF_TOKEN']        = "..."  #Add the API token you hace gotten from IndicTrans2 Huggingface
    print('✅ HF_TOKEN loaded.')
except Exception:
    print('⚠️  HF_TOKEN not found in secrets — IndicTrans2 may fail on gated model.')

try:
    os.environ['GOOGLE_API_KEY']  = "..."   #Add Gemini API key here
    print('✅ GOOGLE_API_KEY loaded.')
except Exception:
    print('⚠️  GOOGLE_API_KEY not found — RAG/Gemini will be disabled, local fallback only.')

# ── Model paths (from your Kaggle model input) ────────────────────────────────
os.environ['YOLO_V8_PATH']  = '/kaggle/input/models/yusufmurtaza01/model/pytorch/default/1/finetuned_yolo8n.pt'
os.environ['YOLO_V12_PATH'] = '/kaggle/input/models/yusufmurtaza01/model/pytorch/default/1/finetuned_yolo12n.pt'

# ── Verify model files exist ──────────────────────────────────────────────────
for name, path in [('YOLOv8n', os.environ['YOLO_V8_PATH']),
                   ('YOLOv12n', os.environ['YOLO_V12_PATH'])]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f'✅ {name}: {path}  ({size_mb:.1f} MB)')
    else:
        print(f'❌ {name} NOT FOUND: {path}')
        print(f'   → Make sure the model dataset is attached in the right panel.')

print('\n✅ Environment ready.')

✅ HF_TOKEN loaded.
✅ GOOGLE_API_KEY loaded.
✅ YOLOv8n: /kaggle/input/models/yusufmurtaza01/model/pytorch/default/1/finetuned_yolo8n.pt  (5.9 MB)
✅ YOLOv12n: /kaggle/input/models/yusufmurtaza01/model/pytorch/default/1/finetuned_yolo12n.pt  (5.3 MB)

✅ Environment ready.


In [3]:
# ── Cell 3: Copy all prerequisite files to working directory ─────────────────
import shutil, os, subprocess, sys

# Kill any leftover Gradio server from a previous run
subprocess.run(['pkill', '-f', 'app.py'], capture_output=True)
subprocess.run(['fuser', '-k', '7860/tcp'], capture_output=True)
subprocess.run(['fuser', '-k', '7861/tcp'], capture_output=True)
import time; time.sleep(2)
print('✅ Cleared any previous Gradio processes.')

PREREQ_DIR  = '/kaggle/input/datasets/yusufmurtaza01/prerequisite'
WORKING_DIR = '/kaggle/working'

FILES_TO_COPY = [
    'app.py',
    'translations.py',
    'disease_id_mapping.py',
    'treatment_data.py',
    'data_split.yaml',
    # 'treatment_data_translated.py',   # ← uncomment after generating on Colab
]

for fname in FILES_TO_COPY:
    src  = os.path.join(PREREQ_DIR, fname)
    dst  = os.path.join(WORKING_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'✅ Copied: {fname}')
    else:
        print(f'❌ Missing: {src}')

# ── Patch launch() in app.py for Kaggle ──────────────────────────────────────
app_path = os.path.join(WORKING_DIR, 'app.py')
with open(app_path) as f:
    src = f.read()

old_launch = '''    demo.queue(max_size=10).launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=False,
        show_error=True,
    )'''

new_launch = '''    demo.queue(max_size=10).launch(
        server_port=7861,    # 7860 is used by HF/Kaggle infra — use 7861
        share=True,          # creates public gradio.live URL
        show_error=True,
    )'''

if old_launch in src:
    src = src.replace(old_launch, new_launch)
    with open(app_path, 'w') as f:
        f.write(src)
    print('✅ Patched launch() → port 7861, share=True, removed server_name')
else:
    print('⚠️  Launch block not found — may already be patched from a previous run, continuing...')

if WORKING_DIR not in sys.path:
    sys.path.insert(0, WORKING_DIR)
os.chdir(WORKING_DIR)
print(f'\n✅ Working directory: {os.getcwd()}')


✅ Cleared any previous Gradio processes.
✅ Copied: app.py
✅ Copied: translations.py
✅ Copied: disease_id_mapping.py
✅ Copied: treatment_data.py
✅ Copied: data_split.yaml
✅ Patched launch() → port 7861, share=True, removed server_name

✅ Working directory: /kaggle/working


In [4]:
# ── Cell 4: Launch the Gradio app ────────────────────────────────────────────
# The public gradio.live URL will appear below — click it to open the app.
# URL is valid for 72 hours.
%run /kaggle/working/app.py

INFO | Pre-warming YOLO models…


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


INFO | ✅ YOLO models loaded successfully.
/kaggle/working/app.py:771: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=THEME, css=CSS, title="Plant Disease Detector") as demo:
/kaggle/working/app.py:771: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=THEME, css=CSS, title="Plant Disease Detector") as demo:
INFO | HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
INFO | HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
INFO | HTTP Request: GET http://127.0.0.1:7861/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO | HTTP Request: HEAD http://127.0.0.1:7861/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7861


INFO | HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
INFO | HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64 "HTTP/1.1 200 OK"


* Running on public URL: https://fec4578f6159feb9ea.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


INFO | HTTP Request: HEAD https://fec4578f6159feb9ea.gradio.live "HTTP/1.1 200 OK"
